<a href="https://colab.research.google.com/github/Koshalendar/Artificial-intelligence-/blob/main/1BM24CS139_week7__unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
def is_variable(expr):
    """
    Determines if an expression is a variable.
    By standard AI convention, variables are single lowercase letters (e.g., 'x', 'y').
    """
    return isinstance(expr, str) and expr.islower() and len(expr) == 1

def is_constant(expr):
    """
    Determines if an expression is a constant.
    By standard AI convention, constants start with an uppercase letter (e.g., 'John', 'A').
    """
    return isinstance(expr, str) and (expr.isupper() or expr[0].isupper())

def occurs_check(var, expr):
    """
    Checks if a variable 'var' occurs inside the expression 'expr'.
    Prevents infinite loops during substitution.
    """
    if var == expr:
        return True
    if isinstance(expr, list):
        # Check inside the arguments of the compound expression
        return any(occurs_check(var, arg) for arg in expr[1:])
    return False

def apply_substitution(subst, expr):
    """
    Applies a substitution set (dictionary) to an expression recursively.
    """
    if not subst:
        return expr
    if is_variable(expr):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [expr[0]] + [apply_substitution(subst, arg) for arg in expr[1:]]
    return expr

def unify(psi1, psi2):
    """
    Main Unification Algorithm following the slide steps.
    """
    # --- Step 1: Handle Variables or Constants ---
    if is_variable(psi1) or is_constant(psi1) or is_variable(psi2) or is_constant(psi2):
        # a) If identical, return NIL (represented as an empty dict {})
        if psi1 == psi2:
            return {}

        # b) Else if psi1 is a variable
        elif is_variable(psi1):
            if occurs_check(psi1, psi2):
                return "FAILURE"
            else:
                return {psi1: psi2}  # Returns { Psi2 / Psi1 }

        # c) Else if psi2 is a variable
        elif is_variable(psi2):
            if occurs_check(psi2, psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}  # Returns { Psi1 / Psi2 }

        # d) Else return FAILURE (e.g., two different constants like 'A' and 'B')
        else:
            return "FAILURE"

    # --- Step 2: Check if Predicate/Function symbols are the same ---
    if psi1[0] != psi2[0]:
        return "FAILURE"

    # --- Step 3: Check if they have a different number of arguments ---
    if len(psi1) != len(psi2):
        return "FAILURE"

    # --- Step 4: Set Substitution set (SUBST) to NIL ---
    subst = {}

    # --- Step 5: For i = 1 to number of elements ---
    # In Python, we iterate through the arguments (index 1 to end)
    for i in range(1, len(psi1)):
        # a) Call Unify with the ith elements
        arg1 = psi1[i]
        arg2 = psi2[i]

        s = unify(arg1, arg2)

        # b) If S = failure then returns Failure
        if s == "FAILURE":
            return "FAILURE"

        # c) If S != NIL then do
        if s != {}:
            # Update the current overall substitution set
            # (Equivalent to appending/merging the new substitution)
            for var in subst:
                subst[var] = apply_substitution(s, subst[var])
            subst.update(s)

            # Apply S to the remainder of both expressions
            psi1 = apply_substitution(s, psi1)
            psi2 = apply_substitution(s, psi2)

    # --- Step 6: Return SUBST ---
    return subst

def parse_input(input_str):
    """
    Helper function to parse a basic string format into a list structure.
    Example: "Knows(John, x)" -> ['Knows', 'John', 'x']
    """
    input_str = input_str.replace(" ", "")
    if "(" not in input_str:
        return input_str # It's a single constant or variable

    # Extract predicate and arguments
    pred, args_str = input_str.split("(", 1)
    args_str = args_str.rstrip(")")

    # Basic top-level splitting for nested functions if any
    # (Simple split by comma works if there are no nested functions)
    args = args_str.split(",")
    return [pred] + args

# --- User Input & Execution ---
if __name__ == "__main__":
    print("--- Unification Algorithm ---")
    print("Rules: Lowercase single letters (x, y, z) are variables.")
    print("       Words starting with Uppercase (John, A, King) are constants/predicates.")
    print("Format example: King(x)  or  Knows(John, x)\n")

    str1 = input("Enter Expression 1 (Psi 1): ")
    str2 = input("Enter Expression 2 (Psi 2): ")

    try:
        psi1 = parse_input(str1)
        psi2 = parse_input(str2)

        result = unify(psi1, psi2)

        print("\n--- Result ---")
        if result == "FAILURE":
            print("❌ Unification Failed!")
        elif result == {}:
            print("✅ Unification Successful! Result: NIL (Expressions already identical)")
        else:
            print("✅ Unification Successful!")
            print("Substitution Set (SUBST):")
            for var, val in result.items():
                # Formats output as { Value / Variable } matching logic textbooks
                # e.g., { John / x }
                val_str = str(val).replace("[", "").replace("]", "").replace("'", "")
                print(f"  {{ {val_str} / {var} }}")

    except Exception as e:
        print(f"\nAn error occurred while parsing. Please check your syntax format. Error: {e}")


--- Unification Algorithm ---
Rules: Lowercase single letters (x, y, z) are variables.
       Words starting with Uppercase (John, A, King) are constants/predicates.
Format example: King(x)  or  Knows(John, x)

Enter Expression 1 (Psi 1):  Q(y,G(A,B))
Enter Expression 2 (Psi 2): Q(G(x,x),y)

--- Result ---
❌ Unification Failed!
